In [6]:
import os
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serverless import ServerlessInferenceConfig
import sagemaker.utils

In [22]:
import os
import sys
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

# --- THE SLEDGEHAMMER FIX ---
import sagemaker.utils
sagemaker.utils = sys.modules['sagemaker.utils']
# ----------------------------

sess = sagemaker.Session()
role = sagemaker.get_execution_role() 

current_dir = os.path.abspath(os.getcwd())
req_path = os.path.join(current_dir, "requirements.txt") 
serve_path = os.path.join(current_dir, "serve.py") 

model_s3_path = "s3://sagemaker-studio-i0gutcxdy/frustration-model/models/model.tar.gz"

print("Configuring the Native Sklearn Ensemble for PROVISIONED Endpoint...")

ensemble_model = SKLearnModel(
    model_data=model_s3_path,
    role=role,
    entry_point=serve_path,
    dependencies=[req_path],
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sess,
    name="frustration-model-pure-v5" 
)

print("Deploying Provisioned endpoint (Using m5.large to bypass the 180s timeout)...")
predictor = ensemble_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="frustration-endpoint-v5", 
    container_startup_health_check_timeout=900 
)

print(f"\n[SUCCESS] Endpoint deployed at: {predictor.endpoint_name}")

Configuring the Native Sklearn Ensemble for PROVISIONED Endpoint...
Deploying Provisioned endpoint (Using m5.large to bypass the 180s timeout)...
-------------------------------------*

Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


UnexpectedStatusException: Error hosting endpoint frustration-endpoint-v5: Failed. Reason: The primary container for production variant AllTraffic did not pass the ping health check. Please check CloudWatch logs for this endpoint.. Try changing the instance type or reference the troubleshooting page https://docs.aws.amazon.com/sagemaker/latest/dg/async-inference-troubleshooting.html